# Underfitting e overfitting

> **Nota AI Lab — ambiente di lettura.** Il testo seguente è la traduzione del notebook Kaggle originale conservato nel repository. I riferimenti a pulsanti, celle nascoste e invii descrivono l’ambiente Kaggle dell’originale; qui le celle di codice sono visibili. Per eseguire i notebook fuori da Kaggle, segui la guida AI Lab all’ambiente.



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=False)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


Alla fine di questa fase comprenderai i concetti di underfitting e overfitting e saprai applicarli per rendere più accurati i tuoi modelli.

# Sperimentare con modelli diversi

Ora che disponi di un modo affidabile per misurare l’accuratezza, puoi sperimentare modelli alternativi e vedere quale produce le previsioni migliori. Ma quali alternative hai?

Nella [documentazione](http://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html) di scikit-learn puoi vedere che l’albero decisionale ha molte opzioni, più di quante ne vorrai o dovrai usare per molto tempo. Le più importanti determinano la profondità dell’albero. Ricorda dalla [prima lezione del corso](https://www.kaggle.com/dansbecker/how-models-work) che la profondità misura il numero di suddivisioni compiute prima di arrivare a una previsione. Questo è un albero relativamente poco profondo:

![Albero di profondità 2](https://storage.googleapis.com/kaggle-media/learn/images/R3ywQsR.png)

Nella pratica non è raro avere dieci suddivisioni tra il livello iniziale, che contiene tutte le case, e una foglia. Più l’albero diventa profondo, più il dataset viene suddiviso in foglie che contengono poche case. Con una sola suddivisione, i dati vengono separati in due gruppi. Dividendo nuovamente ogni gruppo, ne otteniamo quattro. Suddividendoli ancora, ne otteniamo otto. Continuando a raddoppiare i gruppi con nuove suddivisioni a ogni livello, al decimo livello avremo \(2^{10}\) gruppi di case: 1.024 foglie.

Distribuendo le case tra molte foglie, in ogni foglia rimangono meno case. Le foglie con pochissime case producono previsioni molto vicine ai valori reali di quelle abitazioni, ma possono dare previsioni molto inaffidabili su nuovi dati, perché ogni previsione si basa soltanto su poche case.

Questo fenomeno si chiama **overfitting**: il modello si adatta quasi perfettamente ai dati di training, ma si comporta male sulla validazione e sugli altri dati nuovi. All’estremo opposto, se rendiamo l’albero molto poco profondo, le case non vengono suddivise in gruppi sufficientemente distinti.

Nel caso estremo di un albero che divide le case in due o quattro gruppi soltanto, ogni gruppo contiene ancora case molto diverse. Le previsioni possono essere molto lontane dai valori reali per la maggior parte delle case, anche sui dati di training; per la stessa ragione saranno cattive anche sulla validazione. Quando un modello non riesce a cogliere distinzioni e schemi importanti nei dati e si comporta male persino sui dati di training, si parla di **underfitting**.

Poiché ci interessa l’accuratezza su dati nuovi, che stimiamo attraverso i dati di validazione, vogliamo trovare il punto di equilibrio tra underfitting e overfitting. Graficamente, cerchiamo il punto più basso della curva rossa di validazione nella figura seguente.

![Underfitting e overfitting](https://storage.googleapis.com/kaggle-media/learn/images/AXSEOfI.png)

# Esempio
Esistono varie possibilità per controllare la profondità di un albero; molte consentono che alcuni percorsi siano più profondi di altri. L’argomento *max_leaf_nodes* offre però un modo molto ragionevole per controllare il rapporto tra overfitting e underfitting. Più foglie consentiamo al modello di creare, più ci spostiamo dalla zona di underfitting a quella di overfitting del grafico.

Possiamo usare una funzione di utilità per confrontare i punteggi MAE ottenuti con valori diversi di *max_leaf_nodes*:


In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    mae = mean_absolute_error(val_y, preds_val)
    return(mae)

I dati vengono caricati in **train_X**, **val_X**, **train_y** e **val_y** usando il codice che hai già visto e che hai già scritto.


In [ ]:
# Qui viene eseguito il caricamento dei dati.
import pandas as pd
    
# Carica i dati.
melbourne_file_path = '../input/melbourne-housing-snapshot/melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path) 
# Escludi le righe con valori mancanti.
filtered_melbourne_data = melbourne_data.dropna(axis=0)
# Scegli il target e le feature.
y = filtered_melbourne_data.Price
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 
                        'YearBuilt', 'Lattitude', 'Longtitude']
X = filtered_melbourne_data[melbourne_features]

from sklearn.model_selection import train_test_split

# Dividi feature e target nei set di training e validazione.
train_X, val_X, train_y, val_y = train_test_split(X, y,random_state = 0)

Possiamo usare un ciclo `for` per confrontare l’accuratezza dei modelli costruiti con valori diversi di *max_leaf_nodes*.


In [ ]:
# Confronta il MAE per diversi valori di max_leaf_nodes.
for max_leaf_nodes in [5, 50, 500, 5000]:
    my_mae = get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    print("Max leaf nodes: %d  \t\t Mean Absolute Error:  %d" %(max_leaf_nodes, my_mae))

Tra le opzioni elencate, 500 è il numero ottimale di foglie.

---

# Conclusione

Ecco ciò che devi ricordare. I modelli possono soffrire di:

- **Overfitting:** cogliere schemi accidentali che non si ripeteranno in futuro, producendo previsioni meno accurate.
- **Underfitting:** non cogliere schemi rilevanti, producendo anche in questo caso previsioni meno accurate.

Usiamo i dati di **validazione**, che non partecipano all’addestramento, per misurare l’accuratezza di un modello candidato. Questo ci permette di provare molti modelli candidati e conservare il migliore.

# Ora tocca a te

Prova a **[ottimizzare il modello che hai costruito](https://prof-davidaffo.github.io/kaggle-learntools/corsi/machine_learning/ex5.html)**.
